# 2026-11-26 Four models on FOODS: seasonal naive, exponential smoothing, ARIMA, SARIMA
Same 30 store × dept series, same 28-day test, one global MASE per model (1-year scale). Seasonal naive 7 is the number to beat.

## Result
- **SARIMA improves by 7.5%** over seasonal naive 7 (global MASE 0.979 vs 1.058). Wins 23 of 30 series.
- **Exponential smoothing improves by 7.5% too** (0.979). Wins 22 of 30. A tie with SARIMA, and about 40 times faster to fit.
- **ARIMA** (weekly diff, 25 Nov): improves by 4.1% (1.015). Wins 18 of 30.
- Both seasonal models learn the weekly shape instead of copying last week. That is the gain.
- One 28-day test window only. Check a second window before choosing between SARIMA and exponential smoothing.

## Step 1: set up

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsforecast.models import AutoARIMA, AutoETS
from statsforecast.arima import arima_string
from statsmodels.tsa.stattools import acf
from m5food import data, plots
plots.set_style()

HORIZON = 28                 # test days
M = 7                        # season length: one week
SCALE_DAYS = 365             # MASE yardstick: last year of training
PLOT_START = '2016-02-01'    # the "last months" view starts here

sales = data.add_readable_names(data.load_sales_foods())
cal = data.load_calendar()
days = data.day_columns(sales)
idx = pd.DatetimeIndex(cal.set_index('d').loc[days, 'date'].values, name='date')
series = sales.groupby(['store', 'dept'])[days].sum().T.set_axis(idx)       # 30 columns: (store, dept)
train, test = series.iloc[:-HORIZON], series.iloc[-HORIZON:]

stores = sorted(series.columns.get_level_values(0).unique())
depts = sorted(series.columns.get_level_values(1).unique())
STATE_COLOR = {'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}
state_of = lambda store: store.split(' store ')[0]
print(f'train: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} days) | '
      f'test: {test.index[0].date()} to {test.index[-1].date()} ({len(test)} days)')

In [ ]:
def grid(lines, title, subtitle, start=None, end=None, weekly=False):
    """30 panels (rows = stores, columns = depts). lines = {legend name: (DataFrame, line style)}.
    start / end = dates to show. weekly=True plots weekly averages (for the whole-series view). Grey = test days."""
    fig = make_subplots(rows=len(stores), cols=len(depts), shared_xaxes=True,
                        column_titles=[f'<b>{d}</b>' for d in depts], row_titles=stores,
                        vertical_spacing=0.012, horizontal_spacing=0.04)
    for r, store in enumerate(stores, start=1):
        for c, dept in enumerate(depts, start=1):
            for name, (df, style) in lines.items():
                y = df[(store, dept)].loc[start:end]
                if weekly:
                    y = y.resample('W').mean()
                line = dict(style)
                if line.get('color') == 'state':
                    line['color'] = STATE_COLOR[state_of(store)]
                fig.add_trace(go.Scatter(x=y.index, y=y, name=name, line=line, mode='lines',
                                         showlegend=(r == 1 and c == 1 and len(lines) > 1),
                                         hovertemplate=f'{store}, {dept}<br>%{{x|%a %d %b %Y}}: %{{y:,.0f}}<extra>{name}</extra>'),
                              row=r, col=c)
    fig.add_vrect(x0=test.index[0], x1=test.index[-1], fillcolor='#e6e6e6', opacity=0.6, layer='below',
                  line_width=0, row='all', col='all')
    fig.update_yaxes(nticks=3, tickfont=dict(size=9))
    fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
    fig.update_layout(title=f'<b>{title}</b><br><span style="font-size:13px;color:#444">{subtitle}</span>',
                      template='simple_white', height=1500, margin=dict(t=110, r=150),
                      legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
    fig.show()

## Step 2: look at the 30 series: whole, then the last months

In [ ]:
grid({'units a day': (series, dict(color='state', width=1.1))},
     'The 30 series, whole period', 'Weekly average of units a day, 2011 to 2016. Grey = the 28 test days. '
     'Colour = state (blue California, orange Texas, green Wisconsin). Each panel has its own scale.', weekly=True)

In [ ]:
grid({'units a day': (series, dict(color='state', width=1.1))},
     'The 30 series, last months',
     f'Units a day, {pd.Timestamp(PLOT_START):%d %b} to {test.index[-1]:%d %b %Y}. Grey = the 28 test days.',
     start=PLOT_START)

## Step 3: seasonal naive 7, the number to beat

In [ ]:
recent = train.iloc[-SCALE_DAYS:]
scale = (recent - recent.shift(M)).abs().mean()          # per series: average miss of "same weekday last week", last year

def mase(forecast, days=slice(None)):
    """MASE per series. forecast in units, same shape as test. days = which test days to score (default all 28)."""
    return (test - forecast).iloc[days].abs().mean() / scale

def vs_snaive(forecast):
    """% better (+) or worse (-) than seasonal naive 7 on the global MASE."""
    return (1 - mase(forecast).mean() / mase(snaive7).mean()) * 100

snaive7 = pd.DataFrame(np.tile(train.iloc[-M:].to_numpy(), (HORIZON // M, 1)), index=test.index, columns=train.columns)
print(f'Seasonal naive 7: global MASE {mase(snaive7).mean():.3f}')

## Step 4: exponential smoothing
- AutoETS (statsforecast, Hyndman's ets): picks error, trend and season type per series by AICc.
- Season length 7, so it can keep the weekly shape. Fitted on raw units.

In [ ]:
ets = pd.DataFrame(index=test.index, columns=train.columns, dtype=float)
ets_type = {}
for col in train.columns:
    m = AutoETS(season_length=M).fit(train[col].to_numpy(dtype=float))
    ets[col] = m.predict(h=HORIZON)['mean']
    ets_type[col] = m.model_['method']
print('ETS types chosen:', pd.Series(ets_type).value_counts().to_dict())
print(f'Exponential smoothing: global MASE {mase(ets).mean():.3f} | vs seasonal naive {vs_snaive(ets):+.1f}%')

## Step 5: ARIMA (as in the 25 Nov notebook)
- Plain ARIMA on the weekly-differenced data, then back to units.

In [ ]:
def undo_weekly_diff(change):
    """Change forecast -> units. Each week = the week before + the forecast change.
    Week 1 builds on the last real training week, week 2 on forecast week 1, and so on."""
    prev, weeks = train[change.columns].iloc[-M:].to_numpy(), []
    for k in range(HORIZON // M):
        prev = prev + change.iloc[M * k: M * (k + 1)].to_numpy()
        weeks.append(prev)
    return pd.DataFrame(np.vstack(weeks), index=test.index, columns=change.columns)

train_wdiff = (train - train.shift(M)).dropna()
arima_change = pd.DataFrame(index=test.index, columns=train.columns, dtype=float)
arima_fits = {}
for col in train.columns:
    arima_fits[col] = AutoARIMA(season_length=1).fit(train_wdiff[col].to_numpy(dtype=float))
    arima_change[col] = arima_fits[col].predict(h=HORIZON)['mean']
arima = undo_weekly_diff(arima_change)
print(f'ARIMA: global MASE {mase(arima).mean():.3f} | vs seasonal naive {vs_snaive(arima):+.1f}%')

## Step 6: SARIMA
- AutoARIMA with season length 7, fitted on raw units. It picks the normal part (p, d, q) and a weekly part (P, D, Q) at lag 7.
- No manual differencing and no back-transform: the model does both itself.

In [ ]:
def fit_sarima(frame):
    """SARIMA per series. Returns the 28-day forecasts in units and the chosen orders."""
    fc, orders = pd.DataFrame(index=test.index, columns=frame.columns, dtype=float), {}
    for col in frame.columns:
        m = AutoARIMA(season_length=M).fit(frame[col].to_numpy(dtype=float))
        fc[col] = m.predict(h=HORIZON)['mean']
        orders[col] = arima_string(m.model_).strip()
    return fc, pd.Series(orders)

sarima, sarima_orders = fit_sarima(train)
print('Most common orders:', ' | '.join(f'{o} x{n}' for o, n in sarima_orders.value_counts().head(5).items()))
print(f'SARIMA: global MASE {mase(sarima).mean():.3f} | vs seasonal naive {vs_snaive(sarima):+.1f}%')

**What the seasonal part adds:** the weekly terms at lag 7 let the model learn the weekly shape from all the past weeks, not copy one week. That takes it from 4.1% (ARIMA) to 7.5% better than seasonal naive. It chose a weekly difference (D = 1) in 14 of 30 series; the other 16 use weekly AR / MA terms only.

## Step 7: the forecast window

In [ ]:
grid({'actual': (series, dict(color='black', width=1.4)),
      'seasonal naive 7': (snaive7, dict(color=plots.PALETTE[0], width=1.4, dash='dot')),
      'exponential smoothing': (ets, dict(color=plots.PALETTE[2], width=1.4, dash='dash')),
      'ARIMA': (arima, dict(color=plots.PALETTE[3], width=1.4, dash='dashdot')),
      'SARIMA': (sarima, dict(color=plots.PALETTE[1], width=1.6))},
     'The forecast window: four models vs actual, 30 series',
     f'Units a day, {test.index[0]:%d %b} to {test.index[-1]:%d %b %Y}. Each panel has its own scale.',
     start=test.index[0])

## Step 8: all four models in one table

In [ ]:
models = {'seasonal naive 7': snaive7, 'exponential smoothing': ets, 'ARIMA': arima, 'SARIMA': sarima}
per_series = pd.DataFrame({n: mase(f) for n, f in models.items()})
table = pd.DataFrame({
    'global MASE': per_series.mean(),
    'vs seasonal naive %': [vs_snaive(f) for f in models.values()],
    'wins vs seasonal naive (of 30)': per_series.lt(per_series['seasonal naive 7'], axis=0).sum(),
    'best of the 4 (of 30)': per_series.idxmin(axis=1).value_counts().reindex(per_series.columns, fill_value=0),
})
table.loc['seasonal naive 7', ['vs seasonal naive %', 'wins vs seasonal naive (of 30)']] = np.nan

colour = ['white'] + ['#c8ecd0' if v > 0 else '#f4c6c6' for v in table['vs seasonal naive %'].iloc[1:]]
fig = go.Figure(go.Table(
    header=dict(values=['<b>model</b>'] + [f'<b>{c}</b>' for c in table.columns], fill_color='#eeeeee', align='left'),
    cells=dict(values=[table.index, table['global MASE'].map('{:.3f}'.format),
                       table['vs seasonal naive %'].map(lambda v: '' if pd.isna(v) else f'{v:+.1f}%'),
                       table['wins vs seasonal naive (of 30)'].map(lambda v: '' if pd.isna(v) else f'{v:.0f}'),
                       table['best of the 4 (of 30)']],
               fill_color=[['white'] * 4, ['white'] * 4, colour, ['white'] * 4, ['white'] * 4],
               align='left', font=dict(size=13), height=28)))
fig.update_layout(title='<b>Four models, same 30 series, same 28 test days</b><br><span style="font-size:12px;color:#555">'
                        'Global MASE = average over 30, 1-year scale. Green = better than seasonal naive 7.</span>',
                  height=260, margin=dict(t=80, l=10, r=10, b=10))
fig.show()

fig = go.Figure(go.Bar(x=table.index, y=table['global MASE'], text=table['global MASE'].map('{:.3f}'.format),
                       textposition='outside', hovertemplate='%{x}<br>MASE %{y:.3f}<extra></extra>',
                       marker_color=[plots.PALETTE[0], plots.PALETTE[2], plots.PALETTE[3], plots.PALETTE[1]]))
fig.add_hline(y=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title='<b>Global MASE per model</b><br><span style="font-size:13px;color:#444">Lower is better.</span>',
                  yaxis=dict(title='average MASE', range=[0, table['global MASE'].max() * 1.25]),
                  template='simple_white', height=420, width=700, margin=dict(t=100))
fig.show()

**What I see**
- SARIMA and exponential smoothing tie: **0.979, both 7.5% better** than seasonal naive 7.
- SARIMA wins more series against seasonal naive (23 vs 22). Exponential smoothing is best of the 4 more often (12 vs 8).
- ARIMA is third: 4.1% better, 18 wins.
- Seasonal naive 7 is never the best of the 4.

## Step 9: diagnostics for one ARIMA fit (separate from accuracy)
- Residual = what the model got wrong on a training day (actual minus its one-step fit).
- Good residuals look like noise: no pattern over time, no ACF spikes, spread centred on 0.

In [ ]:
ONE = ('California store 3', 'Food dept 3')            # the biggest series
res = pd.Series(arima_fits[ONE].model_['residuals'], index=train_wdiff.index)
band = 1.96 / np.sqrt(len(res))
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    r_acf = acf(res, nlags=28)[1:]
lags = np.arange(1, 29)

fig = make_subplots(rows=2, cols=2, specs=[[{'colspan': 2}, None], [{}, {}]], vertical_spacing=0.14,
                    subplot_titles=['<b>Residuals over time</b>', '<b>Residual ACF</b> (red = weekly lags)',
                                    '<b>Residual spread</b>'])
fig.add_trace(go.Scatter(x=res.index, y=res, mode='lines', line=dict(color='#5b7a99', width=0.8), showlegend=False,
                         hovertemplate='%{x|%d %b %Y}: %{y:,.0f}<extra></extra>'), row=1, col=1)
fig.add_hline(y=0, line=dict(color='#333', width=1), row=1, col=1)
fig.add_trace(go.Bar(x=lags, y=r_acf, showlegend=False, hovertemplate='lag %{x}: %{y:.2f}<extra></extra>',
                     marker_color=np.where(np.isin(lags, [7, 14, 21, 28]), '#c0504d', '#5b7a99')), row=2, col=1)
fig.add_hrect(y0=-band, y1=band, fillcolor='#bbbbbb', opacity=0.35, line_width=0, row=2, col=1)
fig.add_trace(go.Histogram(x=res, nbinsx=60, marker_color='#5b7a99', showlegend=False,
                           hovertemplate='%{x}: %{y} days<extra></extra>'), row=2, col=2)
fig.update_xaxes(tickvals=[1, 7, 14, 21, 28], row=2, col=1)
fig.update_layout(title=f'<b>Diagnostics: ARIMA on weekly diff, {ONE[0]}, {ONE[1]}</b><br>'
                        f'<span style="font-size:13px;color:#444">{arima_string(arima_fits[ONE].model_).strip()}. '
                        'Grey band = noise level.</span>', template='simple_white', height=720, margin=dict(t=110))
fig.show()
print(f'Residual mean {res.mean():.1f}, std {res.std():.0f} units | ACF lags outside the band: '
      f'{[int(l) for l in lags[np.abs(r_acf) > band]]}')

**What I see**
- Residuals centre on 0 (mean 3.5 units) and spread about ±400 units a day.
- But the residual ACF has 12 lags above noise, including 7. Not clean noise.
- So this plain ARIMA leaves a pattern behind, mostly weekly. That fits SARIMA doing better.
- Next: the same check on SARIMA and exponential smoothing (Hyndman 5.4 and 9).

# Appendix
### A1: by dept and by state

In [ ]:
groups = {'by dept': per_series.index.get_level_values('dept'),
          'by state': per_series.index.get_level_values('store').map(state_of)}
colors = {'seasonal naive 7': plots.PALETTE[0], 'exponential smoothing': plots.PALETTE[2],
          'ARIMA': plots.PALETTE[3], 'SARIMA': plots.PALETTE[1]}
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=[f'<b>{g}</b>' for g in groups])
for j, key in enumerate(groups.values(), start=1):
    avg = per_series.groupby(key).mean()
    for name in models:
        fig.add_trace(go.Bar(x=avg.index, y=avg[name], name=name, showlegend=(j == 1), marker_color=colors[name],
                             text=avg[name].map('{:.2f}'.format), textposition='outside', textfont=dict(size=9)),
                      row=1, col=j)
    fig.add_hline(y=1, line=dict(color='#333', dash='dot'), row=1, col=j)
fig.update_yaxes(title='average MASE', range=[0, per_series.groupby(groups['by dept']).mean().max().max() * 1.25])
fig.update_layout(title='<b>Average MASE by dept and by state</b>', barmode='group', bargap=0.2,
                  template='simple_white', height=480, margin=dict(t=90),
                  legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'))
fig.show()
print('vs seasonal naive %, by dept:')
by_dept = per_series.groupby(groups['by dept']).mean()
(1 - by_dept.div(by_dept['seasonal naive 7'], axis=0)).mul(100).drop(columns='seasonal naive 7').round(1)

**What I see**
- Food dept 1 and 2: SARIMA gains most (10.5% and 8.0%).
- Food dept 3 (the biggest): exponential smoothing gains most (8.4%). SARIMA only 2.3%.
- So no single winner everywhere. The tie in the global number hides a split by dept.

### A2: error by test week

In [ ]:
weeks = {f'week {k + 1}': slice(7 * k, 7 * (k + 1)) for k in range(HORIZON // M)}
by_week = pd.DataFrame({n: {w: mase(f, sl).mean() for w, sl in weeks.items()} for n, f in models.items()})
fig = go.Figure([go.Scatter(x=list(weeks), y=by_week[n], name=n, mode='lines+markers', line=dict(color=colors[n], width=2))
                 for n in models])
fig.update_layout(title='<b>MASE by test week</b><br><span style="font-size:13px;color:#444">Average over 30 series. '
                        'Weeks 2 and 3 hold the SNAP days.</span>', yaxis_title='average MASE',
                  template='simple_white', height=420, margin=dict(t=100),
                  legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'))
fig.show()
by_week.round(2)

**What I see**
- Weeks 1 and 4: exponential smoothing is best (0.66 and 0.80).
- Weeks 2 and 3 (SNAP): SARIMA is best (1.13 and 1.19).
- All four still struggle in weeks 2 and 3. None of them knows about SNAP.

### A3: SARIMA on 5, 3 and 2 years of training
- Same SARIMA. Only the training window changes: the last 5, 3 or 2 years before the test.
- The MASE scale stays the same (last training year). Slow: about 5 minutes per window.

In [ ]:
windows = {'5 years': 5 * 365, '3 years': 3 * 365, '2 years': 2 * 365}
sarima_by_window = {name: fit_sarima(train.iloc[-n:])[0] for name, n in windows.items()}

win = {'full (5.2 years, main)': sarima, **sarima_by_window}
win_table = pd.DataFrame({'global MASE': {n: mase(f).mean() for n, f in win.items()},
                          'vs seasonal naive %': {n: vs_snaive(f) for n, f in win.items()},
                          'wins vs seasonal naive (of 30)': {n: int((mase(f) < mase(snaive7)).sum()) for n, f in win.items()}})
win_table.round(3)

**What I see**
- Every window beats seasonal naive by 6.7% to 8.8%.
- 5 years: 0.965 (8.8%), 2 years: 0.966 (8.7%), 3 years: 0.987 (6.7%). Full 5.2 years: 0.979 (7.5%).
- No clear trend with window length. The differences are small for one test window.
- Keep the full window as the main result.